# Appendix A — Frontier Topics

*Reflection, trajectory learning and preference data. Read with skepticism.*

## Why this is an appendix

Reflection and self-improvement papers are full of plausible mechanisms that fail to replicate or fail to ship. The literature is genuinely unsettled. We include this material so readers know it exists. We do **not** recommend wiring any of it into production without an audit trail and a measurable error budget.

Each section below labels itself **Frontier** or **Open Problem**.

In [ ]:
from glassloop.reasoning import EntryType, Scratchpad, TrustLevel

## 1. Reflection as a hypothesis

**Frontier.** A reflection loop has the agent (a) produce a candidate answer, (b) critique it and (c) revise. The critique is itself model output and is **not** evidence. Treat reflection as a way to surface candidate failures, not as proof of correctness.

In [ ]:
def first_attempt() -> Scratchpad:
    pad = Scratchpad()
    pad.add_claim('The overdraft fee is $35', evidence='overdraft.txt', trust=TrustLevel.HIGH)
    pad.add_claim('Customers can dispute fees within 60 days')  # no evidence!
    return pad

def reflect(pad: Scratchpad) -> dict:
    '''Returns a structured critique — what to retry, with reasons.'''
    unsupported = pad.unsupported_claims()
    return {
        'failure_type': 'missing_evidence' if unsupported else None,
        'unsupported_claims': [e.text for e in unsupported],
        'validated': False,  # critic is not ground truth
    }

draft = first_attempt()
critique = reflect(draft)
print(critique)

## 2. When reflection helps

Reflection catches missing-evidence claims because the structural invariant is verifiable in code. The agent can use the critique to retry — this time, attaching evidence to every claim.

In [ ]:
def revised_attempt() -> Scratchpad:
    pad = Scratchpad()
    pad.add_claim('The overdraft fee is $35', evidence='overdraft.txt', trust=TrustLevel.HIGH)
    pad.add_claim('Customers can dispute fees within 60 days', evidence='disputes.txt', trust=TrustLevel.HIGH)
    return pad

revised = revised_attempt()
print('unsupported now:', revised.unsupported_claims())

## 3. When reflection drifts

**Open problem.** When the critic is wrong, the reflection loop produces confident garbage. A critic that says *"looks fine"* on a bad draft does not flag the error. A critic that says *"add citation"* may cause the agent to fabricate one.

Reflection is a candidate failure detector, never a final verdict. The runtime governance gates from Chapter 12 — schema, policy, plausibility — are checked **in addition to**, never instead of, reflection.

In [ ]:
def fake_critic_says_ok(pad):
    return {'failure_type': None, 'unsupported_claims': [], 'validated': True}

draft_with_bug = first_attempt()
print('real check finds:', [e.text for e in draft_with_bug.unsupported_claims()])
print('bad critic says: ', fake_critic_says_ok(draft_with_bug))

## 4. Trajectory preferences

**Frontier.** Preference learning at the trajectory level compares two paths the agent could have taken. A *better* trajectory is one of:
- safer (fewer denials, fewer escalations from missed catches)
- cheaper (fewer tokens, fewer tool calls)
- better grounded (higher coverage on the claim-evidence map)
- more complete (fewer dropped subtasks)
- more policy-compliant

The data shape is just `(trajectory_winner, trajectory_loser)`. Production preference learning on agent trajectories requires datasets larger than tutorial readers can practically run.

In [ ]:
preference_pair = {
    'winner': {'tool_calls': 4, 'tokens': 800, 'evidence_coverage': 0.9},
    'loser':  {'tool_calls': 7, 'tokens': 1500, 'evidence_coverage': 0.6},
}
print(preference_pair)

## Anti-patterns flagged here

- Cargo-cult reflection loops.
- Treating the reflection output as ground truth.
- Trajectory preference learning on tiny datasets.

In [ ]:
# Self-check
assert critique['failure_type'] == 'missing_evidence'
assert revised.unsupported_claims() == []
print('OK')